<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **APIs REST, HTTP, `requests` y JSON**

## *Automatización conectada con servicios externos*

En este laboratorio aprenderemos a conectar Python con servicios externos mediante HTTP.

La secuencia será:

> **petición → respuesta → validación → JSON → transformación → error → reintento → auditoría**

Trabajaremos con:

- HTTP;
- GET;
- POST;
- parámetros;
- headers;
- códigos de estado;
- JSON;
- timeouts;
- autenticación conceptual;
- errores;
- reintentos;
- paginación;
- transformación a DataFrame;
- consumo seguro de APIs.

## Objetivos

Al finalizar deberías poder:

1. Explicar qué es una API REST.
2. Diferenciar cliente y servidor.
3. Entender request y response.
4. Utilizar `requests.get()`.
5. Utilizar parámetros `params`.
6. Leer códigos HTTP.
7. Interpretar JSON.
8. Convertir respuestas a DataFrame.
9. Utilizar `POST`.
10. Enviar datos con `json=`.
11. Utilizar headers.
12. Aplicar timeouts.
13. Capturar errores de conexión.
14. Utilizar `raise_for_status()`.
15. Implementar reintentos sencillos.
16. Entender paginación.
17. Diseñar un pipeline API → procesamiento → salida.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_DISPONIBLE = True
except ImportError:
    REQUESTS_DISPONIBLE = False

print(
    "requests disponible:",
    REQUESTS_DISPONIBLE
)

# 2. ¿Qué es una API?

Una API permite que dos sistemas intercambien información.

Ejemplo conceptual:

```text
Python
  ↓ request
API externa
  ↓ response
JSON
```

Nuestro código actúa como **cliente** y el servicio remoto como **servidor**.

# 3. Componentes básicos de HTTP

In [ ]:
conceptos_http = pd.DataFrame({
    "concepto": [
        "URL",
        "Método",
        "Headers",
        "Params",
        "Body",
        "Status code",
        "Response"
    ],
    "ejemplo": [
        "https://api.ejemplo.com/items",
        "GET / POST",
        "Authorization: Bearer ...",
        "?page=2",
        '{"nombre": "A"}',
        "200 / 404 / 500",
        "JSON"
    ]
})

conceptos_http

# 4. Métodos HTTP más habituales

In [ ]:
metodos = pd.DataFrame({
    "metodo": [
        "GET",
        "POST",
        "PUT",
        "PATCH",
        "DELETE"
    ],
    "uso_habitual": [
        "Leer",
        "Crear",
        "Reemplazar",
        "Modificar parcialmente",
        "Eliminar"
    ]
})

metodos

# 5. Primera petición GET

Usaremos una API pública de ejemplo cuando haya conexión.

Si la red no está disponible, el notebook continuará utilizando datos simulados.

In [ ]:
URL_EJEMPLO = (
    "https://jsonplaceholder.typicode.com/posts"
)

respuesta_get = None

if REQUESTS_DISPONIBLE:

    try:
        respuesta_get = requests.get(
            URL_EJEMPLO,
            timeout=10
        )

        print(
            "Status:",
            respuesta_get.status_code
        )

    except requests.RequestException as error:
        print(
            type(error).__name__,
            "->",
            error
        )

# 5.1 Inspeccionar respuesta

In [ ]:
if respuesta_get is not None:

    print(
        "Content-Type:",
        respuesta_get.headers.get(
            "Content-Type"
        )
    )

    print(
        "Bytes:",
        len(
            respuesta_get.content
        )
    )

### Interpretación

Antes de procesar una respuesta conviene inspeccionar:

- código de estado;
- tipo de contenido;
- tamaño;
- estructura.

# 6. Códigos de estado HTTP

In [ ]:
codigos = pd.DataFrame({
    "rango": [
        "2xx",
        "3xx",
        "4xx",
        "5xx"
    ],
    "significado": [
        "Éxito",
        "Redirección",
        "Error del cliente",
        "Error del servidor"
    ],
    "ejemplos": [
        "200, 201, 204",
        "301, 302",
        "400, 401, 403, 404, 429",
        "500, 502, 503"
    ]
})

codigos

# 7. Convertir JSON

In [ ]:
datos_api = None

if respuesta_get is not None:

    try:
        respuesta_get.raise_for_status()

        datos_api = (
            respuesta_get.json()
        )

        print(
            "Tipo:",
            type(
                datos_api
            )
        )

        print(
            "Registros:",
            len(
                datos_api
            )
        )

    except requests.RequestException as error:
        print(
            type(error).__name__,
            "->",
            error
        )

    except ValueError as error:
        print(
            "JSON inválido:",
            error
        )

# 7.1 Fallback sin conexión

In [ ]:
if datos_api is None:

    datos_api = [
        {
            "userId": 1,
            "id": 1,
            "title": "Informe mensual",
            "body": "Contenido de ejemplo"
        },
        {
            "userId": 1,
            "id": 2,
            "title": "Informe trimestral",
            "body": "Segundo ejemplo"
        },
        {
            "userId": 2,
            "id": 3,
            "title": "Resumen operativo",
            "body": "Tercer ejemplo"
        }
    ]

    print(
        "Usando fallback local."
    )

# 8. JSON: lista de diccionarios

In [ ]:
datos_api[:2]

### Interpretación

Una respuesta JSON muy habitual es:

```text
lista
  ↓
diccionarios
  ↓
DataFrame
```

# 9. Convertir a DataFrame

In [ ]:
df_posts = pd.DataFrame(
    datos_api
)

df_posts.head()

# 9.1 Inspeccionar estructura

In [ ]:
print(
    df_posts.shape
)

print(
    df_posts.dtypes
)

df_posts.head()

# 9.2 Visualizar registros por usuario

In [ ]:
if "userId" in df_posts.columns:

    posts_usuario = (
        df_posts[
            "userId"
        ]
        .value_counts()
        .sort_index()
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        posts_usuario.index.astype(str),
        posts_usuario.values
    )

    plt.xlabel(
        "Usuario"
    )

    plt.ylabel(
        "Posts"
    )

    plt.title(
        "Registros recibidos por usuario"
    )

    plt.tight_layout()
    plt.show()

# 10. Parámetros de consulta

Muchas APIs permiten filtrar mediante query parameters.

Ejemplo:

```text
/posts?userId=1
```

Con `requests`:

```python
requests.get(url, params={"userId": 1})
```

In [ ]:
respuesta_filtrada = None

if REQUESTS_DISPONIBLE:

    try:
        respuesta_filtrada = requests.get(
            URL_EJEMPLO,
            params={
                "userId":
                    1
            },
            timeout=10
        )

        print(
            "URL final:",
            respuesta_filtrada.url
        )

        print(
            "Status:",
            respuesta_filtrada.status_code
        )

    except requests.RequestException as error:
        print(
            type(error).__name__,
            error
        )

# 10.1 Procesar filtrado

In [ ]:
if respuesta_filtrada is not None:

    try:
        respuesta_filtrada.raise_for_status()

        filtrados = (
            respuesta_filtrada.json()
        )

        print(
            "Registros:",
            len(
                filtrados
            )
        )

    except Exception as error:
        print(
            type(error).__name__,
            error
        )

# 11. Headers

In [ ]:
headers_demo = {
    "Accept":
        "application/json",

    "User-Agent":
        "CursoDataGathery/1.0"
}

headers_demo

Los headers pueden transportar:

- formato;
- autenticación;
- idioma;
- versión;
- información del cliente.

# 12. Autenticación conceptual

Patrones frecuentes:

```python
headers = {
    "Authorization": "Bearer TOKEN"
}
```

o:

```python
params = {
    "api_key": "TOKEN"
}
```

Nunca conviene hardcodear secretos dentro del notebook.

Más adelante utilizaremos variables de entorno.

# 13. POST: enviar datos

In [ ]:
URL_POST = (
    "https://jsonplaceholder.typicode.com/posts"
)

payload = {
    "title":
        "Informe generado",

    "body":
        "Contenido automático",

    "userId":
        10
}

payload

# 13.1 Ejecutar POST

In [ ]:
respuesta_post = None

if REQUESTS_DISPONIBLE:

    try:
        respuesta_post = requests.post(
            URL_POST,
            json=payload,
            timeout=10
        )

        print(
            "Status:",
            respuesta_post.status_code
        )

        print(
            respuesta_post.json()
        )

    except requests.RequestException as error:
        print(
            type(error).__name__,
            error
        )

### `json=` vs `data=`

Cuando enviamos JSON, `json=payload` es normalmente la opción correcta.

`requests` serializa automáticamente el diccionario.

# 14. Provocar un 404

In [ ]:
if REQUESTS_DISPONIBLE:

    try:
        respuesta_404 = requests.get(
            "https://jsonplaceholder.typicode.com/ruta_que_no_existe",
            timeout=10
        )

        print(
            "Status:",
            respuesta_404.status_code
        )

    except requests.RequestException as error:
        print(
            type(error).__name__,
            error
        )

# 14.1 `raise_for_status()`

In [ ]:
if REQUESTS_DISPONIBLE:

    try:
        respuesta_404.raise_for_status()

    except requests.HTTPError as error:
        print(
            type(error).__name__,
            "->",
            error
        )

### Interpretación

`raise_for_status()` convierte códigos HTTP de error en excepciones.

Eso evita seguir procesando una respuesta fallida como si fuera válida.

# 15. Timeouts

In [ ]:
def get_seguro(
    url,
    timeout=5
):
    if not REQUESTS_DISPONIBLE:
        return None

    try:
        respuesta = requests.get(
            url,
            timeout=timeout
        )

        respuesta.raise_for_status()

        return respuesta

    except requests.Timeout:
        print(
            "Timeout."
        )

    except requests.ConnectionError:
        print(
            "Error de conexión."
        )

    except requests.HTTPError as error:
        print(
            "Error HTTP:",
            error
        )

    except requests.RequestException as error:
        print(
            "Error requests:",
            error
        )

    return None

### Regla profesional

Evita peticiones sin timeout.

Una llamada de red podría quedar esperando indefinidamente.

# 16. Reintentos sencillos

In [ ]:
def get_con_reintentos(
    url,
    intentos=3,
    espera=1,
    timeout=5
):
    if not REQUESTS_DISPONIBLE:
        return None

    ultimo_error = None

    for intento in range(
        1,
        intentos + 1
    ):

        try:
            respuesta = requests.get(
                url,
                timeout=timeout
            )

            respuesta.raise_for_status()

            print(
                f"Éxito en intento {intento}"
            )

            return respuesta

        except requests.RequestException as error:
            ultimo_error = error

            print(
                f"Intento {intento} fallido:",
                type(error).__name__
            )

            if intento < intentos:
                time.sleep(
                    espera
                )

    print(
        "Todos los intentos fallaron:",
        ultimo_error
    )

    return None

# 16.1 Ejecutar reintento

In [ ]:
respuesta_reintento = (
    get_con_reintentos(
        URL_EJEMPLO,
        intentos=2,
        espera=0.5
    )
)

print(
    respuesta_reintento is not None
)

# 17. Respuesta JSON inesperada

In [ ]:
respuesta_simulada = {
    "status":
        "ok",

    "data":
        {
            "items":
                [
                    {
                        "id":
                            1,
                        "valor":
                            10
                    },
                    {
                        "id":
                            2,
                        "valor":
                            20
                    }
                ]
        }
}

respuesta_simulada

## 17.1 Extraer estructura anidada

In [ ]:
items = (
    respuesta_simulada
    .get(
        "data",
        {}
    )
    .get(
        "items",
        []
    )
)

pd.DataFrame(
    items
)

### Interpretación

No todas las APIs devuelven una lista directamente.

Es habitual encontrar estructuras anidadas.

# 18. Validar esquema mínimo

In [ ]:
def validar_items_api(
    items,
    campos_obligatorios
):
    errores = []

    if not isinstance(
        items,
        list
    ):
        return [
            "La respuesta no es una lista."
        ]

    for indice, item in enumerate(
        items
    ):
        faltantes = (
            set(
                campos_obligatorios
            )
            - set(
                item.keys()
            )
        )

        if faltantes:
            errores.append(
                f"Registro {indice}: faltan {sorted(faltantes)}"
            )

    return errores


validar_items_api(
    items,
    [
        "id",
        "valor"
    ]
)

# 18.1 Provocar inconsistencia de esquema

In [ ]:
items_problematicos = [
    {
        "id": 1,
        "valor": 10
    },
    {
        "id": 2
    }
]

validar_items_api(
    items_problematicos,
    [
        "id",
        "valor"
    ]
)

Primero detectamos el cambio de esquema antes de intentar construir el pipeline.

# 19. Paginación conceptual

Muchas APIs no devuelven todos los registros en una sola respuesta.

Patrones habituales:

```text
?page=1
?page=2
?page=3
```

o:

```json
{
  "items": [...],
  "next": "url..."
}
```

# 19.1 Simular páginas

In [ ]:
paginas_simuladas = {
    1: [
        {
            "id": 1,
            "valor": 10
        },
        {
            "id": 2,
            "valor": 20
        }
    ],

    2: [
        {
            "id": 3,
            "valor": 30
        },
        {
            "id": 4,
            "valor": 40
        }
    ],

    3: []
}

paginas_simuladas

## 19.2 Recorrer hasta página vacía

In [ ]:
todos = []
pagina = 1

while True:

    datos_pagina = (
        paginas_simuladas
        .get(
            pagina,
            []
        )
    )

    print(
        "Página",
        pagina,
        "->",
        len(
            datos_pagina
        )
    )

    if not datos_pagina:
        break

    todos.extend(
        datos_pagina
    )

    pagina += 1

pd.DataFrame(
    todos
)

# 20. Función genérica API → DataFrame

In [ ]:
def json_a_dataframe(
    datos_json
):
    if isinstance(
        datos_json,
        list
    ):
        return pd.DataFrame(
            datos_json
        )

    if isinstance(
        datos_json,
        dict
    ):
        return pd.DataFrame(
            [
                datos_json
            ]
        )

    raise TypeError(
        "JSON debe ser list o dict."
    )


json_a_dataframe(
    [
        {
            "a":
                1
        },
        {
            "a":
                2
        }
    ]
)

# 21. Auditoría de llamadas

In [ ]:
auditoria_api = []

def registrar_llamada(
    metodo,
    url,
    status,
    segundos,
    registros=None,
    error=None
):
    auditoria_api.append({
        "metodo":
            metodo,

        "url":
            url,

        "status":
            status,

        "segundos":
            segundos,

        "registros":
            registros,

        "error":
            error
    })

# 21.1 Llamada auditada

In [ ]:
if REQUESTS_DISPONIBLE:

    inicio = time.perf_counter()

    try:
        r = requests.get(
            URL_EJEMPLO,
            timeout=10
        )

        duracion = (
            time.perf_counter()
            - inicio
        )

        registros = None

        if r.ok:
            try:
                registros = len(
                    r.json()
                )
            except Exception:
                pass

        registrar_llamada(
            "GET",
            r.url,
            r.status_code,
            duracion,
            registros=registros
        )

    except requests.RequestException as error:

        registrar_llamada(
            "GET",
            URL_EJEMPLO,
            None,
            time.perf_counter()
            - inicio,
            error=str(
                error
            )
        )

pd.DataFrame(
    auditoria_api
)

# 22. Pipeline API → transformación → CSV

In [ ]:
def procesar_posts(
    datos
):
    df = pd.DataFrame(
        datos
    )

    columnas = [
        columna
        for columna
        in [
            "userId",
            "id",
            "title"
        ]
        if columna
        in df.columns
    ]

    return df[
        columnas
    ].copy()


posts_limpios = procesar_posts(
    datos_api
)

posts_limpios.head()

## 22.1 Guardar salida

In [ ]:
raiz = Path(
    "demo_api_pipeline"
)

raiz.mkdir(
    exist_ok=True
)

ruta_salida = (
    raiz
    / "posts.csv"
)

posts_limpios.to_csv(
    ruta_salida,
    index=False
)

print(
    ruta_salida.exists()
)

# 23. Visualizar pipeline

In [ ]:
pipeline = pd.DataFrame({
    "paso": [
        1,
        2,
        3,
        4,
        5,
        6
    ],

    "fase": [
        "Request",
        "Validar HTTP",
        "JSON",
        "Validar esquema",
        "Transformar",
        "Exportar"
    ]
})

pipeline

### Patrón profesional

```text
request
   ↓
status
   ↓
raise_for_status
   ↓
json
   ↓
schema validation
   ↓
transform
   ↓
persist
```

# 24. Mini-práctica integrada

Construye un consumidor de API que:

1. haga GET;
2. use `params`;
3. use headers;
4. tenga timeout;
5. capture errores;
6. use `raise_for_status`;
7. convierta JSON;
8. valide campos;
9. transforme a DataFrame;
10. filtre datos;
11. exporte CSV;
12. registre tiempo;
13. registre status code;
14. implemente reintentos;
15. simule paginación.

# 25. Checklist

- [ ] ¿La URL es correcta?
- [ ] ¿El método HTTP es correcto?
- [ ] ¿Uso timeout?
- [ ] ¿Compruebo status?
- [ ] ¿Uso `raise_for_status()`?
- [ ] ¿Capturo errores de red?
- [ ] ¿Valido que la respuesta sea JSON?
- [ ] ¿Valido el esquema?
- [ ] ¿Los secretos están fuera del código?
- [ ] ¿Uso params correctamente?
- [ ] ¿Gestiono paginación?
- [ ] ¿Tengo reintentos cuando aportan valor?
- [ ] ¿Registro tiempos?
- [ ] ¿Registro errores?
- [ ] ¿Persisto la salida?

# 26. Ejercicios finales

### Ejercicio 1
Haz un GET.

### Ejercicio 2
Imprime status code.

### Ejercicio 3
Inspecciona headers.

### Ejercicio 4
Convierte JSON.

### Ejercicio 5
Crea DataFrame.

### Ejercicio 6
Usa `params`.

### Ejercicio 7
Usa headers.

### Ejercicio 8
Haz POST.

### Ejercicio 9
Provoca 404.

### Ejercicio 10
Utiliza `raise_for_status()`.

### Ejercicio 11
Añade timeout.

### Ejercicio 12
Captura `ConnectionError`.

### Ejercicio 13
Implementa reintentos.

### Ejercicio 14
Valida campos obligatorios.

### Ejercicio 15
Simula paginación.

### Ejercicio 16
Registra auditoría.

### Ejercicio 17
Exporta CSV.

### Ejercicio 18
Reto: construye un pipeline API → DataFrame → validación → archivo.

# 27. Preguntas de reflexión

1. ¿Qué es una API?
2. ¿Qué diferencia hay entre GET y POST?
3. ¿Qué representa un status code?
4. ¿Qué aporta JSON?
5. ¿Qué diferencia hay entre params y body?
6. ¿Qué función tienen los headers?
7. ¿Por qué usar timeout?
8. ¿Qué hace `raise_for_status()`?
9. ¿Cuándo tiene sentido reintentar?
10. ¿Por qué validar el esquema?
11. ¿Qué problema resuelve la paginación?
12. ¿Dónde guardarías una API key?
13. ¿Qué registrarías de cada llamada?
14. ¿Cómo conecta este laboratorio con automatización de informes?

# 28. Cierre

Ya sabemos consumir servicios externos.

La siguiente progresión será:

```text
API
 ↓
datos
 ↓
transformación
 ↓
Excel / PDF
```

y después compararemos esta vía con otra forma de extracción:

```text
web scraping
 ↓
BeautifulSoup
```